# Phase 4 - Model Optimization

Validation is used for calibration comparison, threshold selection, error analysis, stability, risk bands, and the assumed intervention scenario. The test metrics in the final section were produced only after those decisions were fixed.

## 1. Load the selected model and validation data

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd() / "loan-default-prediction", Path.cwd().parent / "loan-default-prediction"]
ROOT = next((candidate for candidate in CANDIDATE_ROOTS if (candidate / "models/best_model.joblib").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Cannot find models/best_model.joblib")
sys.path.insert(0, str(ROOT))
from scripts.optimize_model import load_original_split, aligned_original_features
from src.models.evaluate import evaluate_model

metadata = json.loads((ROOT / "models/best_model_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ROOT / "models/best_model.joblib")
validation = pd.read_csv(ROOT / "data/processed/validation.csv")
raw_validation = load_original_split("validation")
preprocessor = joblib.load(ROOT / "data/processed/preprocessor.joblib")
X_validation, y_validation = aligned_original_features(raw_validation, validation, preprocessor)
validation_probability = model.predict_proba(X_validation)[:, 1]
threshold = metadata["classification_threshold"]
selected_name = metadata["model_name"]
calibrated = metadata["calibrated"]
print(f"Model: {selected_name}; calibrated: {calibrated}")
print(f"Validation rows: {len(validation):,}; operating threshold: {threshold:.2f}")

## 2. Validation metrics

In [ ]:
validation_metrics = evaluate_model(model, X_validation, y_validation, threshold=threshold)
for key in ["ROC_AUC", "PR_AUC", "Accuracy", "Precision", "Recall", "F1", "Confusion_Matrix"]:
    print(f"{key}: {validation_metrics[key]}")

## 3. Threshold analysis

In [ ]:
thresholds = pd.read_csv(ROOT / "reports/threshold_analysis.csv")
display(thresholds)
print("Selected threshold:", threshold)

## 4. Validation ROC curve

In [ ]:
false_positive_rate, true_positive_rate, _ = roc_curve(y_validation, validation_probability)
validation_auc = roc_auc_score(y_validation, validation_probability)
plt.figure(figsize=(7, 5))
plt.plot(false_positive_rate, true_positive_rate, label=f"ROC-AUC={validation_auc:.3f}")
plt.plot([0, 1], [0, 1], "--", color="grey")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation ROC curve")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Validation precision-recall curve

In [ ]:
precision, recall, _ = precision_recall_curve(y_validation, validation_probability)
average_precision = average_precision_score(y_validation, validation_probability)
plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"Average Precision={average_precision:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation precision-recall curve")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Calibration curve

In [ ]:
calibration_report = json.loads((ROOT / "reports/calibration_metrics.json").read_text(encoding="utf-8"))
calibrated_candidate = joblib.load(ROOT / "models/calibrated_model.joblib")
candidate_probability = calibrated_candidate.predict_proba(X_validation)[:, 1]
series = {"Selected": validation_probability, "Best calibrated candidate": candidate_probability}
for label, probabilities in series.items():
    observed, predicted = calibration_curve(y_validation, probabilities, n_bins=10, strategy="quantile")
    plt.plot(predicted, observed, marker="o", label=label)
plt.plot([0, 1], [0, 1], "--", color="grey", label="Ideal")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed default frequency")
plt.title("Validation calibration curve")
plt.legend()
plt.tight_layout()
plt.show()
display(pd.DataFrame(calibration_report["validation_metrics"]).T)

## 7. Validation error analysis

In [ ]:
errors = pd.read_csv(ROOT / "reports/error_analysis.csv")
grouped_errors = pd.read_csv(ROOT / "reports/error_analysis_by_feature.csv")
display(errors["Error_Type"].value_counts().rename_axis("Error_Type").to_frame("Count"))
display(grouped_errors.sort_values("False_Negative_Count", ascending=False).head(20))

## 8. Risk bands and NPA scenario

In [ ]:
risk_bands = pd.read_csv(ROOT / "reports/risk_band_analysis.csv")
npa_scenario = json.loads((ROOT / "reports/npa_simulation.json").read_text(encoding="utf-8"))
display(risk_bands)
display(pd.Series(npa_scenario, name="Scenario value").to_frame())
print("SIMULATED / ASSUMED only; not measured real-world NPA reduction.")

## 9. Validation stability checks

In [ ]:
stability = pd.read_csv(ROOT / "reports/stability_analysis.csv")
display(stability)

## 10. Final test evaluation

In [ ]:
test_metrics = metadata["test_metrics"]
print("Final test metrics; the test set was not used for calibration, threshold selection, or model selection.")
for key in ["ROC_AUC", "PR_AUC", "Accuracy", "Precision", "Recall", "F1", "Brier_Score", "Confusion_Matrix"]:
    print(f"{key}: {test_metrics[key]}")
print("Selected validation threshold:", metadata["classification_threshold"])